# Notebook 2 - directory conversion, options, and dependencies

This notebook expands examples `03_convert_directory.py` and `04_custom_options.py`. It converts the entire sample SQL tree, visualizes emitted action types, and demonstrates non-default strategies such as flat layout, safe MERGE conversion, tags, and external declarations.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

from sql2sqlx import ConversionOptions, Layout, MergeStrategy, convert_directory, convert_string

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists()
)
EXAMPLES = ROOT / "examples"

## Convert the sample tree

`convert_directory` mirrors the CLI workflow. Passing no output directory keeps the notebook side-effect-free while still returning every generated SQLX file in memory.

In [ ]:
result = convert_directory(str(EXAMPLES / "sql"))
summary = [(file.relpath, file.action_type.value, file.action_name) for file in result.files]
summary[:10], len(summary)

## Visualize the action mix

Action-type charts make it easy to review how many inputs were converted into tables, views, incrementals, operations, and declarations.

In [ ]:
counts = Counter(file.action_type.value for file in result.files)
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(counts.keys(), counts.values(), color="#59A14F")
ax.set_title("Action types emitted from examples/sql")
ax.set_ylabel("files")
ax.tick_params(axis="x", rotation=20)
plt.show()

## Use conversion options deliberately

The MERGE strategy is opt-in. When enabled, the converter only lifts MERGE statements that satisfy its shape checks and records warnings that document the assumptions the generated Dataform incremental action relies on.

In [ ]:
options = ConversionOptions(
    merge_strategy=MergeStrategy.INCREMENTAL_WHEN_SAFE,
    declare_external=True,
    layout=Layout.FLAT,
    tags=["migrated"],
    annotate=False,
)
option_result = convert_directory(str(EXAMPLES / "sql"), options=options)
[
    (file.relpath, file.action_type.value, file.action_name)
    for file in option_result.files
    if "customer" in file.action_name
]

In [ ]:
customer_attrs = next(file for file in option_result.files if file.action_name == "customer_attrs")
print(customer_attrs.content)

## External declarations

When `declare_external=True`, unresolved reads can become declaration actions so downstream SQLX can depend on external warehouse objects explicitly.

In [ ]:
external = convert_string(
    "CREATE TABLE marts.fx AS SELECT * FROM ext_finance.currency_rates;",
    ConversionOptions(declare_external=True),
)
[(file.relpath, file.action_type.value, file.action_name) for file in external.files]

In [ ]:
for file in external.files:
    print(f"--- {file.relpath} ---")
    print(file.content)